"""
MEPS real-data analysis with four methods in the Figure 3 / Figure 4 style.

Methods:
  1. SCP: traditional split conformal on MEPS 2021 minority calibration scores.
  2. RSA-CP (score-only OT): generate score-only reference scores from MEPS 2020
     score distributions, transport them to the MEPS 2021 score distribution, and
     augment the real calibration scores.
  3. SPI: the repo's regression SPI quantile logic, using MEPS 2020 as the
     additional/reference data.
  4. Generated-only: split conformal calibrated only on MEPS 2020/reference data.

This script does not train a new model. It uses the existing CQR prediction
interval endpoints in data/meps/models/quantile_regression.
"""


In [30]:
!git clone --depth 1 https://github.com/Meshiba/spi.git

'git' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
from __future__ import annotations

import ast
import math
import os
import random
import re
import sys
import time
from functools import lru_cache
from math import comb
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont


In [6]:
# =========================
# Setup paths
# =========================
PROJECT_ROOT = Path(__file__).resolve().parents[1]
CODE_DIR = Path(__file__).resolve().parent
RESULTS_DIR = PROJECT_ROOT / "results" / "meps_four_methods_fig3_fig4"
FIGURES_DIR = RESULTS_DIR / "figures"
MAIN_FIGURES_DIR = FIGURES_DIR / "main"
APPENDIX_FIGURES_DIR = FIGURES_DIR / "appendix"
SPI_ROOT = PROJECT_ROOT / "spi"

if str(SPI_ROOT) not in sys.path:
    sys.path.insert(0, str(SPI_ROOT))

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")



NameError: name '__file__' is not defined

In [22]:
# =========================
# Setup paths (Jupyter-safe)
# =========================
from pathlib import Path
import os
import sys

# Current notebook working directory
CODE_DIR = Path.cwd()

# Adjust this depending on where the notebook is located
PROJECT_ROOT = CODE_DIR.parent

RESULTS_DIR = PROJECT_ROOT / "results" / "meps_four_methods_fig3_fig4"
FIGURES_DIR = RESULTS_DIR / "figures"
MAIN_FIGURES_DIR = FIGURES_DIR / "main"
APPENDIX_FIGURES_DIR = FIGURES_DIR / "appendix"
SPI_ROOT = PROJECT_ROOT / "spi"

# Create directories if they do not exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MAIN_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
APPENDIX_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(SPI_ROOT) not in sys.path:
    sys.path.insert(0, str(SPI_ROOT))

In [14]:
print("CODE_DIR =", CODE_DIR)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("RESULTS_DIR =", RESULTS_DIR)
print("SPI_ROOT =", SPI_ROOT)

CODE_DIR = C:\Users\bhap2601\RSA CP MEPS data
PROJECT_ROOT = C:\Users\bhap2601
RESULTS_DIR = C:\Users\bhap2601\results\meps_four_methods_fig3_fig4
SPI_ROOT = C:\Users\bhap2601\spi


In [24]:
from pathlib import Path
import sys, os

# THIS should be your actual project folder
PROJECT_ROOT = Path(r"C:\Users\bhap2601\RSA CP MEPS data")

# notebook location
CODE_DIR = Path.cwd()

RESULTS_DIR = PROJECT_ROOT / "results" / "meps_four_methods_fig3_fig4"
FIGURES_DIR = RESULTS_DIR / "figures"
MAIN_FIGURES_DIR = FIGURES_DIR / "main"
APPENDIX_FIGURES_DIR = FIGURES_DIR / "appendix"
SPI_ROOT = PROJECT_ROOT / "spi"

# create folders
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MAIN_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
APPENDIX_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# add SPI package
if str(SPI_ROOT) not in sys.path:
    sys.path.insert(0, str(SPI_ROOT))

# sanity checks
print("PROJECT_ROOT =", PROJECT_ROOT)
print("config exists =", (PROJECT_ROOT / "config_files").exists())
print("spi exists =", SPI_ROOT.exists())
print(
    "yaml exists =",
    (PROJECT_ROOT / "config_files" / "meps_regression_ages_0_to_20.yml").exists()
)

PROJECT_ROOT = C:\Users\bhap2601\RSA CP MEPS data
config exists = False
spi exists = False
yaml exists = False


In [26]:
from pathlib import Path

base = Path(r"C:\Users\bhap2601")

for p in base.rglob("spi"):
    print(p)

In [16]:
# =========================
# Experiment settings
# =========================
age_configs = {
    "0-20": "meps_regression_ages_0_to_20.yml",
    "20-40": "meps_regression_ages_20_to_40.yml",
    "40-60": "meps_regression_ages_40_to_60.yml",
    "60-100": "meps_regression_ages_60_to_100.yml",
}

alpha_list = [0.05, 0.10]
available_cqr_alphas = [0.05, 0.10]

main_n_cal = 15
main_n_cal_maj = 1000
n_cal_maj_list = [250, 500, 750, 1000, 1250, 1500, 1750, 2000, 2250, 2500]
n_score_synth_policy = "match_n_cal_maj"
n_test_requested = 500
n_seeds = 50
quick_n_seeds = 20
seed = 1
beta = 0.4
coverage_tolerance = 0.01

rsa_score_generators = [
    "majority_ot",
    "bootstrap_majority_ot",
    "empirical_like_majority_ot",
]

method_order = [
    "SCP",
    "RSA-CP (score-only OT)",
    "SPI",
    "Generated-only",
]

method_colors = {
    "SCP": (31, 119, 180),
    "RSA-CP (score-only OT)": (214, 39, 40),
    "SPI": (44, 160, 44),
    "Generated-only": (255, 127, 14),
}

method_labels = {
    "SCP": "SCP",
    "RSA-CP (score-only OT)": "RSA-CP",
    "SPI": "SPI",
    "Generated-only": "Gen-only",
}


# =========================
# Config and data helpers
# =========================
def strip_inline_comment(value: str) -> str:
    out = []
    in_single = False
    in_double = False
    for char in value:
        if char == "'" and not in_double:
            in_single = not in_single
        elif char == '"' and not in_single:
            in_double = not in_double
        elif char == "#" and not in_single and not in_double:
            break
        out.append(char)
    return "".join(out).strip()


def parse_scalar(value: str):
    value = strip_inline_comment(value)
    if value == "":
        return []
    try:
        return ast.literal_eval(value)
    except Exception:
        pass
    low = value.lower()
    if low == "true":
        return True
    if low == "false":
        return False
    try:
        if "." in value:
            return float(value)
        return int(value)
    except Exception:
        return value


def fallback_load_config(config_path: Path) -> Tuple[dict, dict]:
    sections: Dict[str, dict] = {}
    current: Optional[str] = None
    for raw_line in config_path.read_text().splitlines():
        if not raw_line.strip() or raw_line.lstrip().startswith("#"):
            continue
        if raw_line[:1].strip() and raw_line.rstrip().endswith(":"):
            current = raw_line.strip()[:-1]
            sections[current] = {}
            continue
        if current is None or ":" not in raw_line:
            continue
        key, value = raw_line.split(":", 1)
        sections[current][key.strip()] = parse_scalar(value)
    return sections["real_data_params"], sections["run_params"]


def resolve_project_path(path_like: str) -> Path:
    path = Path(path_like)
    if path.is_absolute():
        return path
    return (PROJECT_ROOT / path).resolve()


def alpha_token(alpha: float) -> str:
    return f"alpha_{alpha:g}"


def nearest_available_cqr_alpha(target_alpha: float) -> float:
    return min(available_cqr_alphas, key=lambda a: (abs(a - target_alpha), a))


def replace_alpha_in_path(path_like: str, target_alpha: float) -> Path:
    text = str(path_like).replace("\\", "/")
    text = re.sub(r"alpha_[0-9.]+", alpha_token(target_alpha), text)
    return resolve_project_path(text)


def load_one_meps_config(age_range: str, target_alpha: float):
    config_path = PROJECT_ROOT / "config_files" / age_configs[age_range]
    real_data_params, run_params = fallback_load_config(config_path)
    base_cqr_alpha = nearest_available_cqr_alpha(target_alpha)

    dataset_path = replace_alpha_in_path(real_data_params["dataset_path"], base_cqr_alpha)
    dataset_maj_path = replace_alpha_in_path(real_data_params["dataset_maj_path"], base_cqr_alpha)

    if not dataset_path.exists() or not dataset_maj_path.exists():
        raise FileNotFoundError(
            f"Missing MEPS prediction directory for age={age_range}, alpha={base_cqr_alpha}: "
            f"{dataset_path}, {dataset_maj_path}"
        )

    X_minority = np.load(dataset_path / "pred.npy").squeeze()
    y_minority = np.load(dataset_path / "true.npy").squeeze()
    X_majority = np.load(dataset_maj_path / "pred.npy").squeeze()
    y_majority = np.load(dataset_maj_path / "true.npy").squeeze()

    run_params = dict(run_params)
    run_params["dataset"] = f"MEPS_alpha_{target_alpha:g}"
    run_params["alpha"] = [float(target_alpha)]
    run_params["age_range"] = age_range

    print("\nLoaded MEPS config")
    print("age_range:", age_range)
    print("config_path:", config_path)
    print("target alpha:", target_alpha)
    print("base CQR endpoint alpha:", base_cqr_alpha)
    print("minority path:", dataset_path)
    print("majority path:", dataset_maj_path)
    print("X_minority shape:", X_minority.shape, "X_majority shape:", X_majority.shape)

    return {
        "config_path": config_path,
        "target_alpha": float(target_alpha),
        "base_cqr_alpha": float(base_cqr_alpha),
        "run_params": run_params,
        "X_minority": prepare_cqr_endpoints(X_minority, target_alpha),
        "y_minority": np.asarray(y_minority, dtype=float).reshape(-1),
        "X_majority": prepare_cqr_endpoints(X_majority, target_alpha),
        "y_majority": np.asarray(y_majority, dtype=float).reshape(-1),
    }


# =========================
# CQR score and CP helpers
# =========================
def prepare_cqr_endpoints(X: np.ndarray, alpha: float) -> np.ndarray:
    X = np.asarray(X, dtype=float).squeeze()
    if X.ndim != 2:
        raise ValueError(f"Expected CQR endpoint/quantile array with 2 dimensions, got {X.shape}.")
    if X.shape[1] == 2:
        return X.astype(float)
    if X.shape[1] > 2:
        grid = (np.arange(X.shape[1]) + 0.5) / X.shape[1]
        lo = int(np.argmin(np.abs(grid - alpha / 2.0)))
        hi = int(np.argmin(np.abs(grid - (1.0 - alpha / 2.0))))
        print(
            f"Using quantile-grid columns {lo}, {hi} for target alpha={alpha:g} "
            f"(levels {grid[lo]:.4f}, {grid[hi]:.4f})."
        )
        return np.column_stack([X[:, lo], X[:, hi]]).astype(float)
    raise ValueError(f"Cannot build CQR endpoints from shape {X.shape}.")


def cqr_scores(intervals: np.ndarray, y: np.ndarray) -> np.ndarray:
    intervals = np.asarray(intervals, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)
    return np.maximum(intervals[:, 0] - y, y - intervals[:, 1])


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    scores = np.sort(np.asarray(scores, dtype=float).reshape(-1))
    n = len(scores)
    if n == 0:
        raise ValueError("No calibration scores supplied.")
    level = (1.0 - float(alpha)) * (1.0 + 1.0 / float(n))
    if level > 1.0:
        return float("inf")
    idx = int(np.ceil(level * n)) - 1
    idx = min(max(idx, 0), n - 1)
    return float(scores[idx])


def evaluate_cqr(base_intervals: np.ndarray, y: np.ndarray, qhat: float) -> Tuple[float, float]:
    base_intervals = np.asarray(base_intervals, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)
    if not np.isfinite(qhat):
        return 1.0, float("inf")
    lower = base_intervals[:, 0] - qhat
    upper = base_intervals[:, 1] + qhat
    coverage = float(np.mean((lower <= y) & (y <= upper)))
    length = float(np.mean(upper - lower))
    return coverage, length


def quantile_interp(scores: np.ndarray, u: np.ndarray) -> np.ndarray:
    scores = np.sort(np.asarray(scores, dtype=float).reshape(-1))
    u = np.asarray(u, dtype=float)
    if len(scores) == 0:
        raise ValueError("Cannot interpolate an empty score array.")
    if len(scores) == 1:
        return np.full_like(u, scores[0], dtype=float)
    grid = (np.arange(len(scores)) + 0.5) / len(scores)
    return np.interp(np.clip(u, 0.0, 1.0), grid, scores, left=scores[0], right=scores[-1])


def ecdf_values(reference_scores: np.ndarray, values: np.ndarray) -> np.ndarray:
    reference_scores = np.sort(np.asarray(reference_scores, dtype=float).reshape(-1))
    values = np.asarray(values, dtype=float).reshape(-1)
    return np.searchsorted(reference_scores, values, side="right") / float(len(reference_scores))


def ot_map_scores(source_scores: np.ndarray, target_scores: np.ndarray, values_to_map: np.ndarray) -> np.ndarray:
    u = ecdf_values(source_scores, values_to_map)
    return quantile_interp(target_scores, u)


def generate_source_scores(
    s_maj: np.ndarray,
    n_score_synth: int,
    run_seed: int,
    generator: str,
) -> np.ndarray:
    s_maj = np.asarray(s_maj, dtype=float).reshape(-1)
    rng = np.random.default_rng(int(run_seed) + 1729)
    n = int(n_score_synth)
    if generator == "majority_ot":
        if n == len(s_maj):
            return s_maj.copy()
        return rng.choice(s_maj, size=n, replace=True)
    if generator == "bootstrap_majority_ot":
        return rng.choice(s_maj, size=n, replace=True)
    if generator == "empirical_like_majority_ot":
        u = rng.uniform(0.0, 1.0, size=n)
        return quantile_interp(s_maj, u)
    raise ValueError(f"Unknown RSA score generator: {generator}")


def qhat_rsa_score_only(
    s_real: np.ndarray,
    s_maj: np.ndarray,
    alpha: float,
    run_seed: int,
    n_score_synth: int,
    generator: str,
) -> float:
    source = generate_source_scores(s_maj, n_score_synth, run_seed, generator)
    transported = ot_map_scores(source_scores=s_maj, target_scores=s_real, values_to_map=source)
    return conformal_quantile(np.concatenate([s_real, transported]), alpha)


# =========================
# SPI regression quantile logic
# =========================
def hg_prob(m: int, n_majority: int, i: int, k: int) -> float:
    return (
        comb(k + i - 2, i - 1)
        * comb(n_majority + m - k - i + 1, m - i)
        / comb(n_majority + m, m)
    )


def hg_cdf(m: int, n_majority: int, i: int, start: int, end: int) -> float:
    total = 0.0
    for k in range(start, end):
        total += hg_prob(m, n_majority, i, k)
    return total


@lru_cache(maxsize=None)
def spi_rank_bounds(n_cal: int, n_majority: int, beta_value: float) -> Tuple[Tuple[int, ...], Tuple[int, ...]]:
    m = int(n_cal) + 1
    n_majority = int(n_majority)
    lower_tail = float(beta_value) / 2.0
    upper_tail = 1.0 - (float(beta_value) - lower_tail)
    r_min = []
    r_max = []

    for row in range(m):
        i = row + 1
        curr_r = 1
        curr_q = hg_cdf(m, n_majority, i, 1, curr_r + 1)
        while curr_q <= lower_tail:
            curr_r += 1
            curr_q += hg_prob(m, n_majority, i, curr_r)
        start = curr_r

        while curr_q < upper_tail:
            curr_r += 1
            curr_q += hg_prob(m, n_majority, i, curr_r)
            if curr_r == n_majority + 1:
                break
        end = curr_r + 1
        r_min.append(start)
        r_max.append(end - 1)

    return tuple(r_min), tuple(r_max)


def qhat_spi(s_real: np.ndarray, s_maj: np.ndarray, alpha: float, beta_value: float) -> float:
    s_real = np.sort(np.asarray(s_real, dtype=float).reshape(-1))
    s_maj = np.sort(np.asarray(s_maj, dtype=float).reshape(-1))
    n_real = len(s_real)
    n_majority = len(s_maj)

    r_min, r_max = spi_rank_bounds(n_real, n_majority, float(beta_value))
    r_min = np.asarray(r_min, dtype=int)
    r_max = np.asarray(r_max, dtype=int)

    level = (1.0 - float(alpha)) * (1.0 + 1.0 / float(n_majority))
    level_plus = (1.0 - float(alpha)) * (
        1.0 + 1.0 / float(n_majority) + 1.0 / float(n_majority * (1.0 - float(alpha)))
    )
    if level_plus > 1.0:
        q_maj_plus = float("inf")
    else:
        idx = int(np.ceil(level_plus * n_majority)) - 1
        idx = min(max(idx, 0), n_majority - 1)
        q_maj_plus = float(s_maj[idx])

    threshold = int(np.ceil(n_majority * level))
    eligible_plus = np.where(r_max <= threshold)[0]
    eligible_minus = np.where(r_min <= threshold)[0]
    r_plus = int(eligible_plus.max() + 1) if len(eligible_plus) else 1
    r_minus = int(eligible_minus.max() + 1) if len(eligible_minus) else 1

    s_r_minus = float("inf") if r_minus > n_real else float(s_real[r_minus - 1])
    s_r_plus = float("inf") if r_plus > n_real else float(s_real[r_plus - 1])
    q = min(q_maj_plus, s_r_minus)
    q = max(q, s_r_plus)
    return float(q)


# =========================
# Experiment core
# =========================
def make_seed_list(n_seeds_used: int) -> List[int]:
    random.seed(seed)
    if n_seeds_used == 1:
        return [seed]
    return random.sample(range(1, 999999), int(n_seeds_used))


def split_indices(
    n_minority: int,
    n_majority: int,
    n_cal: int,
    n_test: int,
    n_cal_maj: int,
    run_seed: int,
) -> Optional[Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    if n_cal + n_test > n_minority:
        print(
            f"WARNING: skip split, need n_cal+n_test={n_cal+n_test}, "
            f"available minority={n_minority}."
        )
        return None
    if n_cal_maj > n_majority:
        print(
            f"WARNING: skip split, need n_cal_maj={n_cal_maj}, "
            f"available majority={n_majority}."
        )
        return None
    rng_real = np.random.default_rng(int(run_seed))
    rng_maj = np.random.default_rng(int(run_seed) + 100003)
    real_perm = rng_real.permutation(n_minority)
    maj_perm = rng_maj.permutation(n_majority)
    idx_cal = real_perm[:n_cal]
    idx_test = real_perm[n_cal : n_cal + n_test]
    idx_maj = maj_perm[:n_cal_maj]
    return idx_cal, idx_test, idx_maj


def n_score_synth_from_n_cal_maj(n_cal_maj: int) -> int:
    if n_score_synth_policy == "match_n_cal_maj":
        return int(n_cal_maj)
    return int(main_n_cal_maj)


def run_one_split(
    data: dict,
    age_range: str,
    alpha: float,
    n_cal: int,
    n_cal_maj: int,
    n_test: int,
    n_seeds_used: int,
    run_seed: int,
) -> pd.DataFrame:
    X_min = data["X_minority"]
    y_min = data["y_minority"]
    X_maj = data["X_majority"]
    y_maj = data["y_majority"]

    split = split_indices(len(X_min), len(X_maj), n_cal, n_test, n_cal_maj, run_seed)
    if split is None:
        return pd.DataFrame()
    idx_cal, idx_test, idx_maj = split

    X_cal, y_cal = X_min[idx_cal], y_min[idx_cal]
    X_test, y_test = X_min[idx_test], y_min[idx_test]
    X_maj_cal, y_maj_cal = X_maj[idx_maj], y_maj[idx_maj]

    s_real = cqr_scores(X_cal, y_cal)
    s_maj = cqr_scores(X_maj_cal, y_maj_cal)
    n_score_synth = n_score_synth_from_n_cal_maj(n_cal_maj)

    rows = []
    qhats = {
        "SCP": conformal_quantile(s_real, alpha),
        "SPI": qhat_spi(s_real, s_maj, alpha, beta),
        "Generated-only": conformal_quantile(s_maj, alpha),
    }

    for method, qhat in qhats.items():
        coverage, length = evaluate_cqr(X_test, y_test, qhat)
        rows.append(
            {
                "experiment": "reference_size_sensitivity",
                "age_range": age_range,
                "alpha": float(alpha),
                "base_cqr_alpha": float(data["base_cqr_alpha"]),
                "n_cal": int(n_cal),
                "n_cal_maj": int(n_cal_maj),
                "n_score_synth": int(n_score_synth),
                "n_test": int(n_test),
                "n_seeds": int(n_seeds_used),
                "seed": int(seed),
                "run_seed": int(run_seed),
                "Method": method,
                "rsa_score_generator": "",
                "Coverage": coverage,
                "Length": length,
                "qhat": float(qhat),
            }
        )

    for generator in rsa_score_generators:
        qhat = qhat_rsa_score_only(
            s_real=s_real,
            s_maj=s_maj,
            alpha=alpha,
            run_seed=run_seed,
            n_score_synth=n_score_synth,
            generator=generator,
        )
        coverage, length = evaluate_cqr(X_test, y_test, qhat)
        rows.append(
            {
                "experiment": "reference_size_sensitivity",
                "age_range": age_range,
                "alpha": float(alpha),
                "base_cqr_alpha": float(data["base_cqr_alpha"]),
                "n_cal": int(n_cal),
                "n_cal_maj": int(n_cal_maj),
                "n_score_synth": int(n_score_synth),
                "n_test": int(n_test),
                "n_seeds": int(n_seeds_used),
                "seed": int(seed),
                "run_seed": int(run_seed),
                "Method": "RSA-CP (score-only OT)",
                "rsa_score_generator": generator,
                "Coverage": coverage,
                "Length": length,
                "qhat": float(qhat),
            }
        )

    return pd.DataFrame(rows)


def run_experiment(n_seeds_used: int) -> pd.DataFrame:
    rows = []
    seed_list = make_seed_list(n_seeds_used)
    print("n_seeds:", n_seeds_used)
    print("seed:", seed)
    print("n_cal:", main_n_cal)
    print("n_cal_maj_list:", n_cal_maj_list)
    print("n_test_requested:", n_test_requested)
    print("alpha_list:", alpha_list)
    print("rsa_score_generators:", rsa_score_generators)

    for age_range in age_configs:
        for alpha in alpha_list:
            data = load_one_meps_config(age_range, alpha)
            n_test = min(int(n_test_requested), len(data["X_minority"]) - int(main_n_cal))
            if n_test <= 0:
                print(f"WARNING: skip age={age_range}, alpha={alpha}: no test samples.")
                continue
            print("Sanity setting:", age_range, alpha, main_n_cal, n_cal_maj_list, n_test, n_seeds_used)
            for n_cal_maj in n_cal_maj_list:
                if n_cal_maj > len(data["X_majority"]):
                    print(f"WARNING: skip n_cal_maj={n_cal_maj}, only {len(data['X_majority'])} available.")
                    continue
                for run_seed in seed_list:
                    curr = run_one_split(
                        data=data,
                        age_range=age_range,
                        alpha=alpha,
                        n_cal=main_n_cal,
                        n_cal_maj=n_cal_maj,
                        n_test=n_test,
                        n_seeds_used=n_seeds_used,
                        run_seed=run_seed,
                    )
                    if not curr.empty:
                        rows.append(curr)

    if not rows:
        raise RuntimeError("No experiment rows were produced.")
    return pd.concat(rows, ignore_index=True)


# =========================
# Summary and selection
# =========================
def summarize(raw: pd.DataFrame) -> pd.DataFrame:
    return (
        raw.groupby(
            [
                "experiment",
                "age_range",
                "alpha",
                "base_cqr_alpha",
                "n_cal",
                "n_cal_maj",
                "n_score_synth",
                "n_test",
                "Method",
                "rsa_score_generator",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            Coverage_mean=("Coverage", "mean"),
            Coverage_std=("Coverage", "std"),
            Length_mean=("Length", "mean"),
            Length_std=("Length", "std"),
            qhat_mean=("qhat", "mean"),
            qhat_std=("qhat", "std"),
            n_runs=("Coverage", "count"),
        )
        .sort_values(["age_range", "alpha", "n_cal_maj", "Method", "rsa_score_generator"])
    )


def select_rsa_generators(summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    rsa = summary[
        (summary["Method"] == "RSA-CP (score-only OT)")
        & (summary["n_cal"] == main_n_cal)
        & (summary["n_cal_maj"] == main_n_cal_maj)
    ].copy()
    for (age_range, alpha), df in rsa.groupby(["age_range", "alpha"], sort=False):
        nominal = 1.0 - float(alpha)
        df = df.assign(
            nominal_coverage=nominal,
            coverage_distance=(df["Coverage_mean"] - nominal).abs(),
            undercoverage=np.maximum(nominal - df["Coverage_mean"], 0.0),
        )
        eligible = df[df["Coverage_mean"] >= nominal - coverage_tolerance].copy()
        if eligible.empty:
            eligible = df.copy()
        chosen = eligible.sort_values(
            ["coverage_distance", "Length_mean", "Coverage_mean"],
            ascending=[True, True, False],
        ).iloc[0]
        rows.append(chosen.to_dict())
    return pd.DataFrame(rows)


def selected_four_method_summary(summary: pd.DataFrame, selection: pd.DataFrame) -> pd.DataFrame:
    parts = [summary[summary["Method"] != "RSA-CP (score-only OT)"].copy()]
    rsa = summary[summary["Method"] == "RSA-CP (score-only OT)"].copy()
    selected_keys = selection[["age_range", "alpha", "rsa_score_generator"]].drop_duplicates()
    rsa_selected = rsa.merge(selected_keys, on=["age_range", "alpha", "rsa_score_generator"], how="inner")
    parts.append(rsa_selected)
    out = pd.concat(parts, ignore_index=True)
    out["Method"] = pd.Categorical(out["Method"], categories=method_order, ordered=True)
    return out.sort_values(["age_range", "alpha", "n_cal_maj", "Method"]).reset_index(drop=True)


def selected_four_method_raw(raw: pd.DataFrame, selection: pd.DataFrame) -> pd.DataFrame:
    parts = [raw[raw["Method"] != "RSA-CP (score-only OT)"].copy()]
    rsa = raw[raw["Method"] == "RSA-CP (score-only OT)"].copy()
    selected_keys = selection[["age_range", "alpha", "rsa_score_generator"]].drop_duplicates()
    parts.append(rsa.merge(selected_keys, on=["age_range", "alpha", "rsa_score_generator"], how="inner"))
    out = pd.concat(parts, ignore_index=True)
    out["Method"] = pd.Categorical(out["Method"], categories=method_order, ordered=True)
    return out.sort_values(["age_range", "alpha", "n_cal_maj", "run_seed", "Method"]).reset_index(drop=True)


def compare_to_scp(selected_summary: pd.DataFrame) -> pd.DataFrame:
    keys = ["age_range", "alpha", "n_cal", "n_cal_maj", "n_score_synth", "n_test"]
    scp = selected_summary[selected_summary["Method"] == "SCP"][
        keys + ["Coverage_mean", "Length_mean"]
    ].rename(columns={"Coverage_mean": "SCP_Coverage_mean", "Length_mean": "SCP_Length_mean"})
    out = selected_summary.merge(scp, on=keys, how="left")
    out["nominal_coverage"] = 1.0 - out["alpha"].astype(float)
    out["coverage_distance"] = (out["Coverage_mean"] - out["nominal_coverage"]).abs()
    out["length_delta_vs_scp"] = out["Length_mean"] - out["SCP_Length_mean"]
    with np.errstate(divide="ignore", invalid="ignore"):
        out["length_pct_vs_scp"] = out["length_delta_vs_scp"] / out["SCP_Length_mean"] * 100.0
    return out.sort_values(["age_range", "alpha", "n_cal_maj", "Method"])


def write_parameter_settings(n_seeds_used: int) -> pd.DataFrame:
    rows = [
        ("Age configs", "age_configs", str(age_configs), "Each config is reloaded before loading data."),
        ("Target alphas", "alpha_list", str(alpha_list), "Uses matching local CQR endpoint alpha where available."),
        ("Available CQR alphas", "available_cqr_alphas", str(available_cqr_alphas), "Local MEPS directories."),
        ("Minority calibration size", "main_n_cal", main_n_cal, "Fixed for Figure 3 and Figure 4."),
        ("Reference sizes", "n_cal_maj_list", str(n_cal_maj_list), "Figure 4 x-axis; SCP does not use this."),
        ("Main reference size", "main_n_cal_maj", main_n_cal_maj, "Figure 3 setting."),
        ("Score synth policy", "n_score_synth_policy", n_score_synth_policy, "RSA score count matches n_cal_maj."),
        ("Requested test size", "n_test_requested", n_test_requested, "Skipped/capped if data are insufficient."),
        ("Number of seeds", "n_seeds", n_seeds_used, "Use --quick for 20."),
        ("Seed", "seed", seed, "Seed generator for reproducible runs."),
        ("SPI beta", "beta", beta, "Matches MEPS SPI config."),
        ("RSA score generators", "rsa_score_generators", str(rsa_score_generators), "Selection table chooses per age/alpha."),
        ("Coverage tolerance", "coverage_tolerance", coverage_tolerance, "Only used for RSA generator selection."),
    ]
    df = pd.DataFrame(rows, columns=["Block", "Parameter", "Value", "Notes"])
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(RESULTS_DIR / "parameter_settings_table.csv", index=False)
    df.to_csv(CODE_DIR / "parameter_settings_four_methods.csv", index=False)
    text = "\n".join([f"{r.Parameter}: {r.Value}  # {r.Notes}" for _, r in df.iterrows()])
    (RESULTS_DIR / "parameter_settings.txt").write_text(text, encoding="utf-8")
    (CODE_DIR / "parameter_settings_four_methods.txt").write_text(text, encoding="utf-8")
    return df


# =========================
# Plotting
# =========================
def font(size: int, bold: bool = False):
    candidates = [
        "C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf",
        "arialbd.ttf" if bold else "arial.ttf",
    ]
    for path in candidates:
        try:
            return ImageFont.truetype(path, size=size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_TITLE = font(26, True)
FONT_PANEL = font(18, True)
FONT = font(15, False)
FONT_SMALL = font(12, False)


def sanitize(value) -> str:
    text = str(value).replace(".", "p")
    return re.sub(r"[^A-Za-z0-9_-]+", "_", text).strip("_")


def save_image_pdf(img: Image.Image, out_base: Path) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_base.with_suffix(".png"))
    img.convert("RGB").save(out_base.with_suffix(".pdf"), "PDF", resolution=120.0)


def finite_limits(values: Sequence[float], nominal: Optional[float] = None, allow_negative: bool = False):
    vals = [float(v) for v in values if np.isfinite(float(v))]
    if nominal is not None and np.isfinite(nominal):
        vals.append(float(nominal))
    if not vals:
        return 0.0, 1.0, 1.0
    lo, hi = min(vals), max(vals)
    if math.isclose(lo, hi):
        pad = max(abs(lo) * 0.08, 0.02)
    else:
        pad = (hi - lo) * 0.18
    lo -= pad
    hi += pad
    if not allow_negative:
        lo = max(0.0, lo)
    if nominal is not None and 0.0 <= nominal <= 1.0:
        hi = min(1.02, hi)
        if hi - lo < 0.05:
            mid = (hi + lo) / 2.0
            lo = max(0.0, mid - 0.025)
            hi = min(1.02, mid + 0.025)
    return lo, hi, hi


def y_pixel(y: float, lo: float, hi: float, top: int, bottom: int) -> float:
    return bottom - (float(y) - lo) / (hi - lo) * (bottom - top)


def draw_axes(draw: ImageDraw.ImageDraw, box, lo, hi):
    left, top, right, bottom = box
    draw.rectangle(box, outline=(210, 210, 210), width=1)
    for i in range(5):
        y = lo + (hi - lo) * i / 4
        yy = y_pixel(y, lo, hi, top, bottom)
        draw.line((left, yy, right, yy), fill=(235, 235, 235), width=1)
        draw.text((left - 62, yy - 8), f"{y:.3g}", fill=(70, 70, 70), font=FONT_SMALL)


def draw_legend(draw: ImageDraw.ImageDraw, x: int, y: int):
    for idx, method in enumerate(method_order):
        yy = y + idx * 26
        draw.line((x, yy + 8, x + 34, yy + 8), fill=method_colors[method], width=4)
        draw.text((x + 44, yy), method_labels[method], fill=(35, 35, 35), font=FONT)


def draw_box_panel(draw: ImageDraw.ImageDraw, raw: pd.DataFrame, box, metric: str, title: str, nominal=None):
    left, top, right, bottom = box
    values = raw[metric].astype(float).values
    lo, hi, cap = finite_limits(values, nominal=nominal)
    draw_axes(draw, box, lo, hi)
    draw.text((left, top - 30), title, fill=(30, 30, 30), font=FONT_PANEL)
    if nominal is not None:
        yy = y_pixel(nominal, lo, hi, top, bottom)
        draw.line((left, yy, right, yy), fill=(0, 0, 0), width=2)
        draw.text((right - 100, yy - 18), f"nominal {nominal:.2f}", fill=(0, 0, 0), font=FONT_SMALL)

    panel_w = right - left
    step = panel_w / (len(method_order) + 1)
    box_w = min(50, step * 0.45)
    for idx, method in enumerate(method_order, start=1):
        vals = raw[raw["Method"] == method][metric].astype(float).values
        if len(vals) == 0:
            continue
        finite = vals[np.isfinite(vals)]
        if len(finite) == 0:
            finite = np.array([cap])
        q1, med, q3 = np.percentile(finite, [25, 50, 75])
        lo_w, hi_w = np.percentile(finite, [5, 95])
        cx = left + step * idx
        color = method_colors[method]
        y_q1 = y_pixel(q1, lo, hi, top, bottom)
        y_q3 = y_pixel(q3, lo, hi, top, bottom)
        y_med = y_pixel(med, lo, hi, top, bottom)
        y_low = y_pixel(lo_w, lo, hi, top, bottom)
        y_high = y_pixel(hi_w, lo, hi, top, bottom)
        draw.line((cx, y_high, cx, y_low), fill=color, width=2)
        draw.rectangle((cx - box_w / 2, y_q3, cx + box_w / 2, y_q1), outline=color, width=2)
        draw.line((cx - box_w / 2, y_med, cx + box_w / 2, y_med), fill=color, width=3)
        draw.text((cx - 28, bottom + 12), method_labels[method], fill=(45, 45, 45), font=FONT_SMALL)


def draw_line_panel(draw: ImageDraw.ImageDraw, summary: pd.DataFrame, box, y_col: str, title: str, nominal=None):
    left, top, right, bottom = box
    values = summary[y_col].astype(float).values
    lo, hi, cap = finite_limits(values, nominal=nominal)
    draw_axes(draw, box, lo, hi)
    draw.text((left, top - 30), title, fill=(30, 30, 30), font=FONT_PANEL)
    xs = sorted(summary["n_cal_maj"].unique())
    x_min, x_max = float(min(xs)), float(max(xs))

    def sx(x):
        if math.isclose(x_min, x_max):
            return (left + right) / 2.0
        return left + (float(x) - x_min) / (x_max - x_min) * (right - left)

    if nominal is not None:
        yy = y_pixel(nominal, lo, hi, top, bottom)
        draw.line((left, yy, right, yy), fill=(0, 0, 0), width=2)
        draw.text((right - 100, yy - 18), f"nominal {nominal:.2f}", fill=(0, 0, 0), font=FONT_SMALL)

    for x in xs:
        xx = sx(x)
        draw.line((xx, bottom, xx, bottom + 6), fill=(40, 40, 40), width=1)
        draw.text((xx - 22, bottom + 12), str(int(x)), fill=(60, 60, 60), font=FONT_SMALL)

    for method in method_order:
        sub = summary[summary["Method"] == method].sort_values("n_cal_maj")
        if sub.empty:
            continue
        pts = []
        for _, row in sub.iterrows():
            y = float(row[y_col])
            if not np.isfinite(y):
                y = cap
            pts.append((sx(row["n_cal_maj"]), y_pixel(y, lo, hi, top, bottom)))
        color = method_colors[method]
        if len(pts) > 1:
            draw.line(pts, fill=color, width=3)
        for x, y in pts:
            draw.ellipse((x - 4, y - 4, x + 4, y + 4), fill=color)

def plot_figure3(raw_selected: pd.DataFrame):
    main_raw = raw_selected[raw_selected["n_cal_maj"] == main_n_cal_maj].copy()
    for (age_range, alpha), df in main_raw.groupby(["age_range", "alpha"], sort=False):
        nominal = 1.0 - float(alpha)
        img = Image.new("RGB", (1650, 760), "white")
        draw = ImageDraw.Draw(img)
        draw.text((55, 28), f"Figure 3 style, MEPS {age_range}, alpha={alpha:g}", fill=(25, 25, 25), font=FONT_TITLE)
        draw.text(
            (55, 64),
            f"n_cal={main_n_cal}, n_cal_maj={main_n_cal_maj}, n_test={int(df['n_test'].iloc[0])}; shared splits across methods",
            fill=(70, 70, 70),
            font=FONT,
        )
        draw_box_panel(draw, df, (95, 150, 760, 610), "Coverage", "Coverage across runs", nominal)
        draw_box_panel(draw, df, (895, 150, 1560, 610), "Length", "Interval length across runs", None)
        draw_legend(draw, 1070, 650)
        save_image_pdf(img, MAIN_FIGURES_DIR / f"figure3_meps_{sanitize(age_range)}_alpha_{sanitize(alpha)}")


def plot_figure4(summary_selected: pd.DataFrame):
    for (age_range, alpha), df in summary_selected.groupby(["age_range", "alpha"], sort=False):
        nominal = 1.0 - float(alpha)
        img = Image.new("RGB", (1650, 760), "white")
        draw = ImageDraw.Draw(img)
        draw.text((55, 28), f"Figure 4 style, MEPS {age_range}, alpha={alpha:g}", fill=(25, 25, 25), font=FONT_TITLE)
        draw.text(
            (55, 64),
            f"Reference-size sensitivity with n_cal fixed at {main_n_cal}; SCP is fixed because it uses no generated/reference data.",
            fill=(70, 70, 70),
            font=FONT,
        )
        draw_line_panel(draw, df, (95, 150, 760, 610), "Coverage_mean", "Coverage mean", nominal)
        draw_line_panel(draw, df, (895, 150, 1560, 610), "Length_mean", "Length mean", None)
        draw.text((685, 710), "n_cal_maj / generated score count", fill=(40, 40, 40), font=FONT)
        draw_legend(draw, 1070, 650)
        save_image_pdf(img, MAIN_FIGURES_DIR / f"figure4_ref_size_meps_{sanitize(age_range)}_alpha_{sanitize(alpha)}")


def make_plots(raw_selected: pd.DataFrame, summary_selected: pd.DataFrame):
    MAIN_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    APPENDIX_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    plot_figure3(raw_selected)
    plot_figure4(summary_selected)
    print("Saved figures:", MAIN_FIGURES_DIR)


# =========================
# Save and print
# =========================
def save_tables(
    raw: pd.DataFrame,
    summary: pd.DataFrame,
    selection: pd.DataFrame,
    raw_selected: pd.DataFrame,
    summary_selected: pd.DataFrame,
    comparison: pd.DataFrame,
) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    raw.to_csv(RESULTS_DIR / "raw_results_all_rsa_generators.csv", index=False)
    summary.to_csv(RESULTS_DIR / "summary_all_rsa_generators.csv", index=False)
    selection.to_csv(RESULTS_DIR / "rsa_score_generator_selection.csv", index=False)
    raw_selected.to_csv(RESULTS_DIR / "raw_results_four_methods_selected.csv", index=False)
    summary_selected.to_csv(RESULTS_DIR / "summary_four_methods_selected.csv", index=False)
    comparison.to_csv(RESULTS_DIR / "summary_four_methods_vs_scp.csv", index=False)

    try:
        with pd.ExcelWriter(RESULTS_DIR / "meps_four_methods_fig3_fig4_tables.xlsx") as writer:
            summary_selected.to_excel(writer, sheet_name="summary_selected", index=False)
            comparison.to_excel(writer, sheet_name="vs_scp", index=False)
            selection.to_excel(writer, sheet_name="rsa_generator_selection", index=False)
            summary.to_excel(writer, sheet_name="summary_all_generators", index=False)
            write_parameter_settings(int(raw["n_seeds"].iloc[0])).to_excel(writer, sheet_name="parameters", index=False)
    except Exception as exc:
        print(f"WARNING: could not save Excel workbook: {exc}")

    print("Saved raw all:", RESULTS_DIR / "raw_results_all_rsa_generators.csv")
    print("Saved selected raw:", RESULTS_DIR / "raw_results_four_methods_selected.csv")
    print("Saved selected summary:", RESULTS_DIR / "summary_four_methods_selected.csv")
    print("Saved comparison:", RESULTS_DIR / "summary_four_methods_vs_scp.csv")
    print("Saved RSA generator selection:", RESULTS_DIR / "rsa_score_generator_selection.csv")


def print_final_tables(summary_selected: pd.DataFrame, comparison: pd.DataFrame, selection: pd.DataFrame) -> None:
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.width", 180)

    print("\nRSA score generator selection:")
    print(
        selection[
            [
                "age_range",
                "alpha",
                "rsa_score_generator",
                "Coverage_mean",
                "Length_mean",
                "n_cal",
                "n_cal_maj",
                "n_runs",
            ]
        ].sort_values(["age_range", "alpha"])
    )

    print("\nFigure 3 setting summary (n_cal=15, n_cal_maj=1000):")
    fig3 = summary_selected[summary_selected["n_cal_maj"] == main_n_cal_maj]
    print(
        fig3[
            [
                "age_range",
                "alpha",
                "Method",
                "rsa_score_generator",
                "Coverage_mean",
                "Coverage_std",
                "Length_mean",
                "Length_std",
                "qhat_mean",
                "n_runs",
            ]
        ].sort_values(["age_range", "alpha", "Method"])
    )

    print("\nFour methods vs SCP:")
    cols = [
        "age_range",
        "alpha",
        "n_cal_maj",
        "Method",
        "rsa_score_generator",
        "Coverage_mean",
        "Length_mean",
        "SCP_Coverage_mean",
        "SCP_Length_mean",
        "length_delta_vs_scp",
        "length_pct_vs_scp",
    ]
    print(comparison[cols].sort_values(["age_range", "alpha", "n_cal_maj", "Method"]))


def main(argv: Optional[Sequence[str]] = None) -> None:
    argv = list(sys.argv[1:] if argv is None else argv)
    n_seeds_used = quick_n_seeds if "--quick" in argv else n_seeds
    if "--quick" in argv:
        print(f"Quick mode enabled: n_seeds={n_seeds_used}")

    t0 = time.time()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    write_parameter_settings(n_seeds_used)

    raw = run_experiment(n_seeds_used)
    summary = summarize(raw)
    selection = select_rsa_generators(summary)
    raw_selected = selected_four_method_raw(raw, selection)
    summary_selected = selected_four_method_summary(summary, selection)
    comparison = compare_to_scp(summary_selected)

    save_tables(raw, summary, selection, raw_selected, summary_selected, comparison)
    make_plots(raw_selected, summary_selected)
    print_final_tables(summary_selected, comparison, selection)
    print(f"\nDone in {time.time() - t0:.2f} seconds.")


In [18]:
if __name__ == "__main__":
    main()


n_seeds: 50
seed: 1
n_cal: 15
n_cal_maj_list: [250, 500, 750, 1000, 1250, 1500, 1750, 2000, 2250, 2500]
n_test_requested: 500
alpha_list: [0.05, 0.1]
rsa_score_generators: ['majority_ot', 'bootstrap_majority_ot', 'empirical_like_majority_ot']


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\bhap2601\\config_files\\meps_regression_ages_0_to_20.yml'